# Edge-aware Tabular + GNN — Validation RMSE Tuning

This notebook tunes the GINE-based edge-aware Tabular + GNN using validation RMSE, restores the best checkpoint, evaluates the held-out test set **once after model selection**, and saves the best model and results.

## Before running
- Upload your original `MolGBN-OPR-master.zip` when prompted.
- Run all cells in order.
- A GPU (Runtime → Change runtime type → T4 GPU) is recommended.
- This notebook is a tuning experiment, not a guarantee that the model will beat TableGraph.
- Hyperparameter trials share one fixed random split. Keep the test set untouched during tuning.


In [ ]:
# 1. Install dependencies
%pip -q install torch-geometric ogb rdkit scikit-learn pandas joblib


In [ ]:
# 2. Upload the original project ZIP
from google.colab import files
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
if not zip_names:
    raise FileNotFoundError("Upload the original MolGBN-OPR-master.zip.")
PROJECT_ZIP = zip_names[0]


In [ ]:
# 3. Extract and locate dataset
import os, zipfile, glob, shutil, copy, json, random, math, itertools, time
from pathlib import Path

EXTRACT_DIR = Path("/content/molgbn_project")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PROJECT_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

dataset_matches = list(EXTRACT_DIR.rglob("MemTrOC-Dataset.csv"))
if not dataset_matches:
    raise FileNotFoundError("MemTrOC-Dataset.csv was not found in the ZIP.")
DATA_PATH = dataset_matches[0]
print("Dataset:", DATA_PATH)


In [ ]:
# 4. Imports and seed
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GINEConv, global_mean_pool
from ogb.utils import smiles2graph
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib

SEED = 41
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", DEVICE)


In [ ]:
# 5. Model definition
ATOM_FEATURE_CARDINALITIES = (119, 4, 12, 12, 10, 6, 6, 2, 2)
BOND_FEATURE_CARDINALITIES = (5, 6, 2)

class CategoricalFeatureEncoder(nn.Module):
    def __init__(self, cardinalities, hidden_dim):
        super().__init__()
        self.cardinalities = tuple(cardinalities)
        self.embeddings = nn.ModuleList([nn.Embedding(n, hidden_dim) for n in self.cardinalities])
    def forward(self, features):
        features = features.long()
        if features.ndim != 2 or features.shape[1] != len(self.embeddings):
            raise ValueError(f"Expected {len(self.embeddings)} feature columns, got {tuple(features.shape)}")
        out = 0
        for i, emb in enumerate(self.embeddings):
            out = out + emb(features[:, i].clamp(0, self.cardinalities[i] - 1))
        return out

class EdgeAwareTabularGNN(nn.Module):
    def __init__(self, table_input_dim, hidden_dim=128, dropout=0.2, num_layers=2):
        super().__init__()
        self.dropout = dropout
        self.table_net = nn.Sequential(
            nn.Linear(table_input_dim, hidden_dim), nn.ReLU(), nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim), nn.ReLU()
        )
        self.atom_encoder = CategoricalFeatureEncoder(ATOM_FEATURE_CARDINALITIES, hidden_dim)
        self.bond_encoder = CategoricalFeatureEncoder(BOND_FEATURE_CARDINALITIES, hidden_dim)
        self.gnn_layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.gnn_layers.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
        self.head = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1)
        )
    def forward(self, table_x, graph):
        table_emb = self.table_net(table_x)
        x = self.atom_encoder(graph.x)
        edge_emb = self.bond_encoder(graph.edge_attr)
        for conv, norm in zip(self.gnn_layers, self.norms):
            update = conv(x, graph.edge_index, edge_emb)
            x = norm(F.relu(update) + x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        graph_emb = global_mean_pool(x, graph.batch)
        return self.head(torch.cat([table_emb, graph_emb], dim=1))


In [ ]:
# 6. Load dataset using the project's expected column layout
df_all = pd.read_csv(DATA_PATH)
if df_all.shape[1] < 24:
    raise ValueError(f"Expected at least 24 columns based on project layout; found {df_all.shape[1]}.")

smiles_col = df_all.columns[3]
feature_cols = list(df_all.columns[4:23])  # 19 tabular features
target_col = df_all.columns[23]            # target rejection percentage

df = df_all[[smiles_col] + feature_cols + [target_col]].copy()
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df[smiles_col] = df[smiles_col].astype("string").str.strip()
df = df.dropna(subset=[smiles_col, target_col])
df = df[df[smiles_col].notna() & df[smiles_col].ne("")].reset_index(drop=True)

smiles = df[smiles_col].astype(str).to_numpy()
X_raw = df[feature_cols].to_numpy(dtype=np.float64)
y = df[target_col].to_numpy(dtype=np.float32)
print("Rows:", len(df), "| Features:", len(feature_cols), "| Target:", target_col)
print("SMILES column:", smiles_col)
display(df[[smiles_col, target_col]].head())


In [ ]:
# 7. One fixed split for all hyperparameter trials; fit preprocessing on train only
all_idx = np.arange(len(df))
train_val_idx, test_idx = train_test_split(all_idx, test_size=0.10, random_state=SEED)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=(0.20 / 0.90), random_state=SEED
)

imputer = SimpleImputer(strategy="median")
scaler = MinMaxScaler()
X_train = scaler.fit_transform(imputer.fit_transform(X_raw[train_idx])).astype(np.float32)
X_val = scaler.transform(imputer.transform(X_raw[val_idx])).astype(np.float32)
X_test = scaler.transform(imputer.transform(X_raw[test_idx])).astype(np.float32)
print(f"Train={len(train_idx)}, validation={len(val_idx)}, test={len(test_idx)}")


In [ ]:
# 8. Convert SMILES to molecular graphs once and cache
graphs = []
bad = []
for i, smi in enumerate(smiles):
    if Chem.MolFromSmiles(smi) is None:
        bad.append((i, smi))
        graphs.append(None)
        continue
    gd = smiles2graph(smi)
    x = torch.as_tensor(gd["node_feat"], dtype=torch.long)
    edge_index = torch.as_tensor(gd["edge_index"], dtype=torch.long)
    edge_attr = torch.as_tensor(gd["edge_feat"], dtype=torch.long)
    if edge_attr.numel() == 0:
        edge_attr = edge_attr.reshape(0, len(BOND_FEATURE_CARDINALITIES))
    graphs.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr))
if bad:
    raise ValueError(f"Invalid SMILES rows: {bad[:10]} (total {len(bad)}). Check dataset before training.")
print("Cached graphs:", len(graphs))


In [ ]:
# 9. Dataset, loader, metrics
class TableGraphDataset(Dataset):
    def __init__(self, X, graph_list, target):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.graph_list = graph_list
        self.y = torch.as_tensor(target, dtype=torch.float32).reshape(-1)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.graph_list[i], self.y[i]

def collate_fn(items):
    xs, gs, ys = zip(*items)
    return torch.stack(xs), Batch.from_data_list(list(gs)), torch.stack(ys).reshape(-1, 1)

train_ds = TableGraphDataset(X_train, [graphs[i] for i in train_idx], y[train_idx])
val_ds = TableGraphDataset(X_val, [graphs[i] for i in val_idx], y[val_idx])
test_ds = TableGraphDataset(X_test, [graphs[i] for i in test_idx], y[test_idx])

def make_loaders(batch_size):
    gen = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0, generator=gen)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0)
    return train_loader, val_loader, test_loader

def metrics(y_true, y_pred):
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred))
    }

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for tx, graph, target in loader:
        pred = model(tx.to(DEVICE), graph.to(DEVICE))
        ys.extend(target.numpy().reshape(-1))
        ps.extend(pred.cpu().numpy().reshape(-1))
    return metrics(np.asarray(ys), np.asarray(ps)), np.asarray(ys), np.asarray(ps)


## Hyperparameter search

The default grid below runs 12 trials. With a GPU and a moderate dataset, this may take a while. For a quick smoke test, set `MAX_EPOCHS = 80`, `PATIENCE = 15`, and reduce `TRIALS` to 2–3. The final chosen configuration is based only on validation RMSE.

In [ ]:
# 10. Search space and training settings
from itertools import product

# A compact grid; increase it only after confirming the pipeline works.
SEARCH_SPACE = list(product(
    [64, 128],          # hidden_dim
    [0.10, 0.25],       # dropout
    [0.0003, 0.001],    # learning rate
    [1, 2],             # GINE layers
    [64]                # batch size
))
# This grid has 16 trials. Use TRIALS=3 for a smoke test.
TRIALS = len(SEARCH_SPACE)
MAX_EPOCHS = 200
PATIENCE = 30
WEIGHT_DECAY = 1e-4
OUTPUT_DIR = Path("/content/edge_aware_gnn_tuned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Trials:", TRIALS)
print("Max epochs per trial:", MAX_EPOCHS, "| Early-stopping patience:", PATIENCE)


In [ ]:
# 11. Run hyperparameter tuning; checkpoint best validation RMSE
trial_rows = []
global_best_rmse = float("inf")
global_best_state = None
global_best_config = None
global_best_epoch = None
global_best_history = None
start_all = time.time()

for trial_no, (hidden_dim, dropout, lr, num_layers, batch_size) in enumerate(SEARCH_SPACE[:TRIALS], start=1):
    trial_seed = SEED  # same seed for fairer comparisons
    seed_everything(trial_seed)
    train_loader, val_loader, _ = make_loaders(batch_size)

    model = EdgeAwareTabularGNN(
        table_input_dim=len(feature_cols),
        hidden_dim=hidden_dim,
        dropout=dropout,
        num_layers=num_layers
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=8
    )
    loss_fn = nn.MSELoss()

    best_trial_rmse = float("inf")
    best_trial_state = None
    best_trial_epoch = 0
    no_improve = 0
    trial_history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        squared_error_sum, count = 0.0, 0
        for tx, graph, target in train_loader:
            tx, graph, target = tx.to(DEVICE), graph.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(tx, graph)
            loss = loss_fn(pred, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            squared_error_sum += loss.item() * len(target)
            count += len(target)

        train_rmse = math.sqrt(squared_error_sum / max(count, 1))
        val_result, _, _ = evaluate(model, val_loader)
        scheduler.step(val_result["RMSE"])
        trial_history.append({
            "trial": trial_no, "epoch": epoch, "train_RMSE": train_rmse,
            "val_R2": val_result["R2"], "val_RMSE": val_result["RMSE"],
            "val_MAE": val_result["MAE"], "lr": optimizer.param_groups[0]["lr"]
        })

        if val_result["RMSE"] < best_trial_rmse:
            best_trial_rmse = val_result["RMSE"]
            best_trial_state = copy.deepcopy(model.state_dict())
            best_trial_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            break

    trial_rows.append({
        "trial": trial_no,
        "hidden_dim": hidden_dim,
        "dropout": dropout,
        "learning_rate": lr,
        "num_layers": num_layers,
        "batch_size": batch_size,
        "best_epoch": best_trial_epoch,
        "best_val_RMSE": best_trial_rmse
    })
    pd.DataFrame(trial_rows).sort_values("best_val_RMSE").to_csv(
        OUTPUT_DIR / "tuning_results.csv", index=False
    )
    pd.DataFrame(trial_history).to_csv(
        OUTPUT_DIR / f"trial_{trial_no:02d}_history.csv", index=False
    )

    print(
        f"Trial {trial_no:02d}/{TRIALS} | hidden={hidden_dim}, dropout={dropout}, "
        f"lr={lr}, layers={num_layers} | best epoch={best_trial_epoch} | "
        f"val RMSE={best_trial_rmse:.4f}"
    )

    if best_trial_rmse < global_best_rmse:
        global_best_rmse = best_trial_rmse
        global_best_state = copy.deepcopy(best_trial_state)
        global_best_config = {
            "hidden_dim": hidden_dim, "dropout": dropout,
            "learning_rate": lr, "num_layers": num_layers,
            "batch_size": batch_size, "weight_decay": WEIGHT_DECAY
        }
        global_best_epoch = best_trial_epoch
        global_best_history = trial_history.copy()

print(f"\nTuning finished in {(time.time()-start_all)/60:.1f} minutes.")
print("Best validation RMSE:", global_best_rmse)
print("Best configuration:", global_best_config)


In [ ]:
# 12. Rebuild the best model and restore the best validation checkpoint
if global_best_state is None:
    raise RuntimeError("No valid model checkpoint was produced.")

best_model = EdgeAwareTabularGNN(
    table_input_dim=len(feature_cols),
    hidden_dim=global_best_config["hidden_dim"],
    dropout=global_best_config["dropout"],
    num_layers=global_best_config["num_layers"]
).to(DEVICE)
best_model.load_state_dict(global_best_state)

_, val_loader, test_loader = make_loaders(global_best_config["batch_size"])
best_val_metrics, val_true, val_pred = evaluate(best_model, val_loader)
print("Best validation metrics:", json.dumps(best_val_metrics, indent=2))


In [ ]:
# 13. Evaluate held-out test once after selecting the configuration
test_metrics, test_true, test_pred = evaluate(best_model, test_loader)
print("Held-out test metrics:", json.dumps(test_metrics, indent=2))

final_results = {
    "model": "EdgeAwareTabularGNN",
    "seed": SEED,
    "best_config": global_best_config,
    "best_epoch": int(global_best_epoch),
    "best_validation_rmse": float(global_best_rmse),
    "validation_metrics": best_val_metrics,
    "test_metrics": test_metrics,
    "split_sizes": {"train": int(len(train_idx)), "validation": int(len(val_idx)), "test": int(len(test_idx))},
    "note": "Hyperparameters selected by validation RMSE. Test set evaluated after selection. Random row split may share SMILES across splits."
}


In [ ]:
# 14. Save best checkpoint and all tuning artifacts
checkpoint_path = OUTPUT_DIR / "best_edge_aware_tabular_gnn.pt"
torch.save({
    "model_state_dict": best_model.state_dict(),
    "model_name": "EdgeAwareTabularGNN",
    "table_input_dim": len(feature_cols),
    "feature_columns": feature_cols,
    "target_column": target_col,
    "smiles_column": smiles_col,
    "config": global_best_config,
    "seed": SEED,
    "best_epoch": int(global_best_epoch),
    "best_validation_rmse": float(global_best_rmse)
}, checkpoint_path)

with open(OUTPUT_DIR / "final_metrics.json", "w") as f:
    json.dump(final_results, f, indent=2)
joblib.dump({"imputer": imputer, "scaler": scaler, "feature_columns": feature_cols},
            OUTPUT_DIR / "preprocessing.joblib")
pd.DataFrame(trial_rows).sort_values("best_val_RMSE").to_csv(OUTPUT_DIR / "tuning_results.csv", index=False)
pd.DataFrame({"y_true": test_true, "y_pred": test_pred, "residual": test_true-test_pred}).to_csv(
    OUTPUT_DIR / "test_predictions.csv", index=False
)
print("Saved:", sorted(p.name for p in OUTPUT_DIR.iterdir()))
display(pd.DataFrame(trial_rows).sort_values("best_val_RMSE"))


In [ ]:
# 15. Zip and download the results
import shutil
zip_path = shutil.make_archive("/content/edge_aware_gnn_tuned_results", "zip", OUTPUT_DIR)
from google.colab import files
files.download(zip_path)


## Notes
- Select models using **validation RMSE**. Do not use test metrics to choose hyperparameters.
- Compare TableGraph and this model on exactly the same split before deciding which is better.
- The current split is a random row split. For performance on unseen molecules, repeat evaluation with groups defined by SMILES so a molecule cannot occur in both train and test.
- The model architecture here is GINE with categorical atom and bond feature embeddings, a tabular MLP, concatenation fusion, and a regression head.
